# Notebook 7 — CORN Ordinal U-Net

**Goal:** Replace the flat 5-class softmax head in Notebook 3 with a **CORN (Conditional Ordinal
Regression for Neural Networks)** head. CORN trains K−1 binary classifiers with a conditional
likelihood that naturally encodes the ordinal ranking, producing well-calibrated probability
distributions across adjacent classes.

**Why CORN over soft-label CE?**
- Soft labels nudge the gradient; CORN restructures the loss entirely around ordinal thresholds.
- Each binary classifier is trained only on the subset of pixels where the label is ≥ threshold,
  avoiding label leakage across non-adjacent classes.
- Expected-value decoding from CORN probabilities directly maximises Accuracy±1.

**HITL gate:** After training, compare val Accuracy±1 for CE / soft-ordinal (NB3) / CORN (this
notebook). If CORN beats NB3 by ≥0.5 pp, include it in the NB10 final stack.

**Prerequisite:** Notebook 3 must have been run (for train-set normalisation statistics).
CORN runs independently — it does **not** require NB3 checkpoint to initialise.

**Outputs:**
- `results/subtask1/checkpoints/corn_unet_best.pt`
- `results/subtask1/val_preds/corn_val_probs.npz` — soft probabilities for NB10 stacking
- `results/subtask1/val_preds/corn_val_metrics.json`
- `results/subtask1/val_vis_corn/` — side-by-side ground-truth vs prediction images

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd

_cwd = Path(os.getcwd())
REPO_ROOT = _cwd if (_cwd / 'scripts').exists() else _cwd.parent

DATA_DIR   = REPO_ROOT / 'data' / 'subtask1'
CKPT_DIR   = REPO_ROOT / 'results' / 'subtask1' / 'checkpoints'
PRED_DIR   = REPO_ROOT / 'results' / 'subtask1' / 'val_preds'
VIS_DIR    = REPO_ROOT / 'results' / 'subtask1' / 'val_vis_corn'
NORM_FILE  = REPO_ROOT / 'results' / 'subtask1' / 'norm_stats.npz'

for d in [CKPT_DIR, PRED_DIR, VIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)

In [ ]:
# ── Config (edit before running) ──────────────────────────────────────────────
# Temporal reduction strategy: 'concat' | 'summary' | 'seasonal'
#   concat   → 340 ch  (all 34 frames × 10 bands)
#   summary  → 20 ch   (per-band temporal mean + std)
#   seasonal → 40 ch   (per-band seasonal means: spring/summer/autumn/winter)
TEMPORAL_MODE = 'summary'

BATCH_SIZE    = 8        # reduce to 4 if OOM on 32 GB VRAM
NUM_EPOCHS    = 30
LR            = 3e-4
BASE_CH       = 32       # U-Net base channels; double to 64 if budget allows
NUM_WORKERS   = 0        # 0 = main process only; safe on all RunPod configs
PATIENCE      = 7        # early stopping on val Acc±1
NUM_CLASSES   = 5
PATCH_SIZE    = 128
T, C          = 34, 10   # timeframes, bands

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio
from rasterio.windows import Window

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. CORN Head and Loss

CORN decomposes `P(Y = k)` into K−1 conditional binary probabilities:

$$P(Y > k \mid Y > k-1) = \sigma(f_k(x))$$

The joint exceedance probability is their cumulative product:

$$P(Y > k) = \prod_{j=0}^{k} P(Y > j \mid Y > j-1)$$

Class probabilities are recovered as differences of adjacent exceedance probabilities.

The loss trains each binary classifier **only on the subset of pixels with label ≥ threshold**,
making the gradients conditional and avoiding class leakage.

In [ ]:
def corn_probs(logits: torch.Tensor) -> torch.Tensor:
    """
    Convert CORN logits to class probabilities.
    logits: (..., K-1) — raw threshold logits
    returns: (..., K) — normalised class probs summing to 1
    """
    cond = torch.sigmoid(logits)                          # P(Y>k | Y>k-1)
    cum  = torch.cumprod(cond, dim=-1)                    # P(Y>k), shape (..., K-1)
    ones  = torch.ones(*logits.shape[:-1], 1, device=logits.device)
    zeros = torch.zeros(*logits.shape[:-1], 1, device=logits.device)
    ext   = torch.cat([ones, cum, zeros], dim=-1)         # (..., K+1)
    probs = ext[..., :-1] - ext[..., 1:]                  # (..., K)
    return probs.clamp(min=0)                              # numerical guard


def corn_loss_spatial(logits: torch.Tensor,
                      targets: torch.Tensor) -> torch.Tensor:
    """
    CORN loss for pixel-level spatial predictions.
    logits:  (B, K-1, H, W)
    targets: (B, H, W)  — integer 0..K-1
    """
    B, K_minus_1, H, W = logits.shape
    num_classes = K_minus_1 + 1

    # Flatten spatial: (N, K-1) and (N,)
    lg_flat = logits.permute(0, 2, 3, 1).reshape(-1, K_minus_1)
    tg_flat = targets.reshape(-1).long()

    loss = lg_flat.new_zeros(1)
    n_thresholds = 0
    for thresh in range(1, num_classes):           # threshold k means Y > thresh-1
        mask = tg_flat >= thresh                   # only pixels where label ≥ thresh
        if mask.sum() == 0:
            continue
        logit_k = lg_flat[mask, thresh - 1]       # (M,)
        label_k = (tg_flat[mask] > thresh - 1).float()  # (M,) binary: Y > thresh-1
        loss += F.binary_cross_entropy_with_logits(logit_k, label_k)
        n_thresholds += 1

    return loss / max(n_thresholds, 1)


def expected_value_decode(probs: torch.Tensor) -> torch.Tensor:
    """
    Accuracy±1-optimal decoding: round expected class value.
    probs: (..., K)
    returns: (...,) long tensor
    """
    K = probs.shape[-1]
    classes = torch.arange(K, dtype=probs.dtype, device=probs.device)
    ev = (probs * classes).sum(dim=-1)
    return ev.round().long().clamp(0, K - 1)


print('CORN functions defined.')

# Quick sanity: probs for a perfect prediction of class 2
_lg = torch.tensor([-5.0, -5.0, 5.0, 5.0]).unsqueeze(0)   # (1, 4)
_pr = corn_probs(_lg)
print('Sanity probs (should peak at class 2):', _pr.squeeze().detach().numpy().round(3))

## 2. Dataset

In [ ]:
def list_sentinel_tifs(data_dir: Path):
    """Return sorted list of Sentinel-2 GeoTIFF files in data_dir."""
    tifs = sorted(data_dir.glob('S2*.tif'))
    if not tifs:
        tifs = sorted(data_dir.glob('**/*.tif'))
    return tifs


class AgriDataset(Dataset):
    """Patch dataset for AgriPotential Subtask 1."""

    def __init__(self, csv_path, data_dir, norm_stats=None,
                 temporal_mode='summary', augment=False):
        self.df          = pd.read_csv(csv_path)
        self.data_dir    = Path(data_dir)
        self.norm_stats  = norm_stats   # dict with 'mean' and 'std', shape (T*C,) or (C,)
        self.temporal_mode = temporal_mode
        self.augment     = augment
        self.sentinel_files = list_sentinel_tifs(self.data_dir)
        if not self.sentinel_files:
            raise FileNotFoundError(f'No Sentinel TIFs found in {self.data_dir}')
        self.has_labels  = 'label_path' in self.df.columns

    def __len__(self):
        return len(self.df)

    def _load_stack(self, row_idx, col_idx, patch_size):
        win    = Window(col_idx, row_idx, patch_size, patch_size)
        frames = []
        for f in self.sentinel_files:
            with rasterio.open(f) as src:
                frames.append(src.read(window=win).astype(np.float32))
        stack = np.stack(frames, axis=0)   # (T, C, H, W)
        return stack / 10000.0             # reflectance in [0, 1]

    def _reduce_temporal(self, stack):
        """stack: (T, C, H, W) → (features, H, W)"""
        if self.temporal_mode == 'concat':
            T, C, H, W = stack.shape
            return stack.reshape(T * C, H, W)
        elif self.temporal_mode == 'summary':
            mean = stack.mean(axis=0)     # (C, H, W)
            std  = stack.std(axis=0)
            return np.concatenate([mean, std], axis=0)   # (2C, H, W)
        elif self.temporal_mode == 'seasonal':
            # Split 34 frames into 4 roughly equal seasons
            T = stack.shape[0]
            q = T // 4
            chunks = [stack[:q], stack[q:2*q], stack[2*q:3*q], stack[3*q:]]
            return np.concatenate([c.mean(0) for c in chunks], axis=0)  # (4C, H, W)
        else:
            raise ValueError(f'Unknown temporal_mode: {self.temporal_mode}')

    def _normalise(self, x):
        if self.norm_stats is None:
            return x
        m = self.norm_stats['mean'][:, None, None].astype(np.float32)
        s = self.norm_stats['std'][:, None, None].astype(np.float32)
        s = np.where(s == 0, 1.0, s)
        return (x - m) / s

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        patch_id = row['patch_id']
        r, c, ps = int(row['row']), int(row['col']), int(row['patch_size'])

        stack    = self._load_stack(r, c, ps)          # (T, C, H, W)
        features = self._reduce_temporal(stack)        # (F, H, W)
        features = self._normalise(features)
        x_tensor = torch.from_numpy(features)          # float32

        if self.has_labels:
            with rasterio.open(row['label_path']) as src:
                label = src.read(1, window=Window(c, r, ps, ps))
            label = torch.from_numpy(label.astype(np.int64))
        else:
            label = torch.zeros(ps, ps, dtype=torch.long)

        if self.augment:
            k = np.random.randint(0, 4)
            x_tensor = torch.rot90(x_tensor, k, dims=[1, 2])
            label    = torch.rot90(label.unsqueeze(0), k, dims=[1, 2]).squeeze(0)
            if np.random.rand() < 0.5:
                x_tensor = torch.flip(x_tensor, dims=[2])
                label    = torch.flip(label.unsqueeze(0), dims=[2]).squeeze(0)

        return x_tensor, label, patch_id

In [ ]:
# Load or compute normalisation statistics
# If NB3 already saved norm_stats.npz, load it; otherwise compute on train set
if NORM_FILE.exists():
    norm_stats = dict(np.load(NORM_FILE))
    print('Loaded norm stats from:', NORM_FILE)
    print('  mean shape:', norm_stats['mean'].shape)
else:
    print('norm_stats.npz not found — will compute on-the-fly (slower).')
    print('Run Notebook 3 first to cache these statistics.')
    norm_stats = None

TRAIN_CSV = DATA_DIR / 'train.csv'
VAL_CSV   = DATA_DIR / 'val.csv'
TEST_CSV  = DATA_DIR / 'test.csv'

for p in [TRAIN_CSV, VAL_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'Missing: {p} — run data download first')

train_ds = AgriDataset(TRAIN_CSV, DATA_DIR, norm_stats, TEMPORAL_MODE, augment=True)
val_ds   = AgriDataset(VAL_CSV,   DATA_DIR, norm_stats, TEMPORAL_MODE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))

# Infer number of input channels from temporal mode
sample_x, _, _ = train_ds[0]
IN_CHANNELS = sample_x.shape[0]
print(f'Train patches: {len(train_ds)}, Val patches: {len(val_ds)}')
print(f'Temporal mode: {TEMPORAL_MODE}, Input channels: {IN_CHANNELS}')

## 3. U-Net with CORN Head

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class CORNUNet(nn.Module):
    """
    Lightweight 4-level U-Net with a CORN ordinal head.
    Output: (B, K-1, H, W) threshold logits (not probabilities).
    """

    def __init__(self, in_channels, num_classes, base_ch=32):
        super().__init__()
        K = num_classes

        # Encoder
        self.enc1 = DoubleConv(in_channels, base_ch)
        self.enc2 = DoubleConv(base_ch,     base_ch * 2)
        self.enc3 = DoubleConv(base_ch * 2, base_ch * 4)
        self.enc4 = DoubleConv(base_ch * 4, base_ch * 8)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base_ch * 8, base_ch * 16)

        # Decoder
        self.up4   = nn.ConvTranspose2d(base_ch * 16, base_ch * 8, 2, stride=2)
        self.dec4  = DoubleConv(base_ch * 16, base_ch * 8)
        self.up3   = nn.ConvTranspose2d(base_ch * 8,  base_ch * 4, 2, stride=2)
        self.dec3  = DoubleConv(base_ch * 8,  base_ch * 4)
        self.up2   = nn.ConvTranspose2d(base_ch * 4,  base_ch * 2, 2, stride=2)
        self.dec2  = DoubleConv(base_ch * 4,  base_ch * 2)
        self.up1   = nn.ConvTranspose2d(base_ch * 2,  base_ch,     2, stride=2)
        self.dec1  = DoubleConv(base_ch * 2,  base_ch)

        # CORN head: K-1 threshold logits per pixel
        self.head = nn.Conv2d(base_ch, K - 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b  = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)     # (B, K-1, H, W)


model = CORNUNet(IN_CHANNELS, NUM_CLASSES, base_ch=BASE_CH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CORNUNet — {n_params/1e6:.2f}M parameters')

# Quick forward pass test
with torch.no_grad():
    dummy = torch.randn(2, IN_CHANNELS, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
    out   = model(dummy)
print(f'Output shape: {tuple(out.shape)}  (expected (2, {NUM_CLASSES-1}, {PATCH_SIZE}, {PATCH_SIZE}))')

## 4. Metrics

In [ ]:
def acc_pm1(pred: np.ndarray, gt: np.ndarray) -> float:
    """Accuracy±1: fraction of pixels where |pred - gt| <= 1."""
    return float((np.abs(pred.astype(int) - gt.astype(int)) <= 1).mean())

def exact_acc(pred: np.ndarray, gt: np.ndarray) -> float:
    return float((pred == gt).mean())

def confusion_matrix_np(pred: np.ndarray, gt: np.ndarray, K=5):
    cm = np.zeros((K, K), dtype=np.int64)
    for t, p in zip(gt.ravel(), pred.ravel()):
        cm[t, p] += 1
    return cm

## 5. Training Loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

best_acc_pm1 = 0.0
patience_ctr = 0
history      = []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(x)                        # (B, K-1, H, W)
            loss   = corn_loss_spatial(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    # ── Validate ──
    model.eval()
    all_pred, all_gt = [], []
    with torch.no_grad():
        for x, y, _ in val_loader:
            x = x.to(DEVICE)
            logits = model(x)                        # (B, K-1, H, W)
            probs  = corn_probs(logits.permute(0,2,3,1))  # (B, H, W, K)
            pred   = expected_value_decode(probs)    # (B, H, W)
            all_pred.append(pred.cpu().numpy())
            all_gt.append(y.numpy())

    all_pred = np.concatenate([p.ravel() for p in all_pred])
    all_gt   = np.concatenate([g.ravel() for g in all_gt])
    val_apm1 = acc_pm1(all_pred, all_gt)
    val_acc  = exact_acc(all_pred, all_gt)

    improved = val_apm1 > best_acc_pm1
    if improved:
        best_acc_pm1 = val_apm1
        patience_ctr = 0
        torch.save({
            'epoch':         epoch,
            'model_state':   model.state_dict(),
            'val_acc_pm1':   val_apm1,
            'temporal_mode': TEMPORAL_MODE,
            'in_channels':   IN_CHANNELS,
            'base_ch':       BASE_CH,
        }, CKPT_DIR / 'corn_unet_best.pt')
    else:
        patience_ctr += 1

    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_acc_pm1': val_apm1, 'val_acc': val_acc})

    star = ' ★' if improved else ''
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  loss={train_loss:.4f}  '
          f'Acc±1={val_apm1:.4f}  Acc={val_acc:.4f}{star}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'\nBest val Acc±1: {best_acc_pm1:.4f}')

In [ ]:
# ── Plot training history ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_df['epoch'], hist_df['train_loss'], label='Train loss')
ax1.set(xlabel='Epoch', ylabel='CORN Loss', title='Training Loss')
ax1.legend()
ax2.plot(hist_df['epoch'], hist_df['val_acc_pm1'], label='Val Acc±1')
ax2.plot(hist_df['epoch'], hist_df['val_acc'],     label='Val Acc (exact)')
ax2.axhline(0.90, color='red', linestyle='--', alpha=0.5, label='Target 0.90')
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Validation Accuracy')
ax2.legend()
plt.tight_layout()
plt.savefig(CKPT_DIR / 'corn_training_history.png', dpi=100)
plt.show()

hist_df.to_csv(CKPT_DIR / 'corn_training_history.csv', index=False)
print('History saved.')

## 6. Final Evaluation on Val Set (Best Checkpoint)

In [ ]:
# Reload best checkpoint
ckpt = torch.load(CKPT_DIR / 'corn_unet_best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]}, val Acc±1={ckpt["val_acc_pm1"]:.4f})')

model.eval()
patch_results = []
all_probs_list, all_gt_list = [], []

with torch.no_grad():
    for x, y, patch_ids in val_loader:
        x = x.to(DEVICE)
        logits = model(x)                            # (B, K-1, H, W)
        probs  = corn_probs(logits.permute(0,2,3,1)) # (B, H, W, K)
        pred   = expected_value_decode(probs)         # (B, H, W)

        for i, pid in enumerate(patch_ids):
            p = pred[i].cpu().numpy()
            g = y[i].numpy()
            patch_results.append({
                'patch_id':    pid,
                'acc_pm1':     acc_pm1(p, g),
                'exact_acc':   exact_acc(p, g),
            })
        all_probs_list.append(probs.cpu().numpy())
        all_gt_list.append(y.numpy())

results_df = pd.DataFrame(patch_results)
all_probs  = np.concatenate(all_probs_list, axis=0)    # (N_val, H, W, K)
all_gt     = np.concatenate(all_gt_list, axis=0)       # (N_val, H, W)
all_pred   = np.argmax(all_probs, axis=-1)             # argmax for confusion matrix
all_pred_ev = np.round(
    (all_probs * np.arange(NUM_CLASSES)).sum(-1)
).astype(int).clip(0, NUM_CLASSES - 1)

print(f'\n=== Val Metrics (CORN U-Net) ===')
print(f'  Acc±1 (argmax):    {acc_pm1(all_pred.ravel(), all_gt.ravel()):.4f}')
print(f'  Acc±1 (EV decode): {acc_pm1(all_pred_ev.ravel(), all_gt.ravel()):.4f}')
print(f'  Exact Acc (EV):    {exact_acc(all_pred_ev.ravel(), all_gt.ravel()):.4f}')

In [ ]:
# Confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix_np(all_pred_ev.ravel(), all_gt.ravel(), K=NUM_CLASSES)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=range(NUM_CLASSES), yticklabels=range(NUM_CLASSES), ax=ax)
ax.set(xlabel='Predicted', ylabel='True', title='CORN U-Net — Normalised Confusion Matrix')
plt.tight_layout()
plt.savefig(PRED_DIR / 'corn_confusion_matrix.png', dpi=100)
plt.show()

# Per-class recall
print('Per-class recall:', cm_norm.diagonal().round(3))

In [ ]:
# Save val soft probabilities for stacking in NB10
np.savez_compressed(
    PRED_DIR / 'corn_val_probs.npz',
    probs   = all_probs,           # (N, H, W, K)
    gt      = all_gt,              # (N, H, W)
    patch_ids = results_df['patch_id'].values,
)

# Save per-patch metrics
metrics = {
    'val_acc_pm1_ev':  float(acc_pm1(all_pred_ev.ravel(), all_gt.ravel())),
    'val_acc_exact_ev': float(exact_acc(all_pred_ev.ravel(), all_gt.ravel())),
    'val_acc_pm1_argmax': float(acc_pm1(all_pred.ravel(), all_gt.ravel())),
    'temporal_mode':   TEMPORAL_MODE,
    'base_ch':         BASE_CH,
    'best_epoch':      int(ckpt['epoch']),
}
(PRED_DIR / 'corn_val_metrics.json').write_text(json.dumps(metrics, indent=2))
print('Saved val probs and metrics.')
print(json.dumps(metrics, indent=2))

## 7. Visual Inspection (5 Val Patches)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image

CLASS_CMAP = ListedColormap(['#1a1a1a', '#4d8c57', '#a3c86a', '#f5e642', '#e07b1a'])
N_VIS = 5

val_ds_vis = AgriDataset(VAL_CSV, DATA_DIR, norm_stats, TEMPORAL_MODE, augment=False)
indices = np.random.choice(len(val_ds_vis), N_VIS, replace=False)

fig, axes = plt.subplots(N_VIS, 3, figsize=(12, N_VIS * 3))
for row_i, idx in enumerate(indices):
    x, y_gt, patch_id = val_ds_vis[idx]
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(DEVICE))           # (1, K-1, H, W)
        probs  = corn_probs(logits.permute(0,2,3,1))        # (1, H, W, K)
        ev_map = (probs.cpu().numpy()[0] * np.arange(NUM_CLASSES)).sum(-1)  # (H, W)
        pred   = np.round(ev_map).astype(int).clip(0, NUM_CLASSES-1)

    gt_np = y_gt.numpy()

    axes[row_i, 0].imshow(gt_np, cmap=CLASS_CMAP, vmin=0, vmax=4, interpolation='nearest')
    axes[row_i, 0].set_title(f'{patch_id} — Ground Truth')
    axes[row_i, 0].axis('off')

    axes[row_i, 1].imshow(pred, cmap=CLASS_CMAP, vmin=0, vmax=4, interpolation='nearest')
    axes[row_i, 1].set_title(f'CORN Prediction  (Acc±1={acc_pm1(pred, gt_np):.3f})')
    axes[row_i, 1].axis('off')

    axes[row_i, 2].imshow(ev_map, cmap='RdYlGn', vmin=0, vmax=4)
    axes[row_i, 2].set_title('Expected value (continuous)')
    axes[row_i, 2].axis('off')

plt.suptitle('CORN U-Net — Val Patch Visualisations', fontsize=14)
plt.tight_layout()
plt.savefig(VIS_DIR / 'corn_val_vis.png', dpi=100)
plt.show()
print('Saved to', VIS_DIR / 'corn_val_vis.png')

## HITL Decision Gate

Review the confusion matrix and training history above, then decide:

| Outcome | Decision |
|---------|----------|
| CORN val Acc±1 ≥ NB3 U-Net by ≥ 0.5 pp | **Include CORN in NB10 ensemble** |
| CORN val Acc±1 ≈ NB3 (within 0.5 pp) | Include with low weight (0.2) in NB10 |
| CORN val Acc±1 < NB3 | Exclude from NB10; check if temporal mode change helps |

**If class 0 or class 4 recall is < 0.50:** Re-run with `BASE_CH=64` or add class-weight
boosting in the CORN loss (multiply loss for minority-class thresholds by 2.0).

**Next:** Proceed to `09_pseudo_labeling.ipynb` to exploit the 800 unlabeled test patches.